# XGBoost

Nesse notebook nós iremos treinar um modelo XGBoost em nosso dataset médico. O objetivo aqui é checar possíveis melhorias usando a abordagem de um modelo baseado árvores de decisão e boosting.

## Importando as Bibliotecas

Primeiro vamos carregar nossas bibliotecas.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Carregando o Dataset

Agora vamos carregar nosso dataset.

In [2]:
df = pd.read_parquet("../../data/processed/UCMF_fitted.parquet")

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 11705 entries, 0 to 12872
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   peso                    9587 non-null   Float64 
 1   altura                  8078 non-null   Int64   
 2   imc                     7708 non-null   Int64   
 3   idade                   10865 non-null  Float64 
 4   pulsos                  11657 non-null  category
 5   pa_sistolica            5131 non-null   Int64   
 6   pa_diastolica           5121 non-null   Int64   
 7   ppa                     10768 non-null  category
 8   patologia               11705 non-null  category
 9   b2                      11674 non-null  category
 10  sopro                   11682 non-null  category
 11  fc                      10986 non-null  Int64   
 12  hda1                    8565 non-null   category
 13  hda2                    11705 non-null  category
 14  sexo                    11701 non-null

## Seleção de Modelo

Para selecionar o melhor modelo de XGBoost vamos otimizar os hiperparâmetros usando a biblioteca `Optuna` que faz Otimização Bayesiana. A escolha do `Optuna` se dá pelo fato de nós otimizarmos muitos hiperparâmetros com diversos valores e essa biblioteca é perfeita para tarefas custosas como essa. Para isso vamos primeiro montar nosso pré-processador.

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import (
    SimpleImputer,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)

def create_preprocessor(X):
    numeric_columns = X.select_dtypes(include="number").columns.to_list()
    categorical_columns = X.select_dtypes(include="category").columns.to_list()

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        # ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric_features", numeric_pipeline, numeric_columns),
            ("categorical_features", categorical_pipeline, categorical_columns),
        ],
        remainder="passthrough"
    )

    numeric_pipeline.set_output(transform="pandas")
    categorical_pipeline.set_output(transform="pandas")
    preprocessor.set_output(transform="pandas")

    return preprocessor

Agora vamos criar a função objetivo que o `Optuna` irá otimizar. Internamente os parâmetros dessa função serão nossos hiperparâmetros e a imagem dela será o $\text{ROC-AUC}$, portanto o `Optuna` tentará achar os parâmetros (hiperparâmetros) que maximizem a imagem ($\text{ROC-AUC}$).

In [16]:
from xgboost import XGBClassifier

random_state = 42

cross_validator = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)

def create_objetive(X, y):
    preprocessor = create_preprocessor(X)
    preprocessor.set_output(transform="pandas")

    X_opa = preprocessor.fit_transform(X)
    print(X_opa.info())

    def objective(trial):
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="aucpr",
            random_state=random_state,
            tree_method="hist",
            enable_categorical=True,
            n_estimators=trial.suggest_int("n_estimators", 200, 2000),
            learning_rate=trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 3, 10),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            gamma=trial.suggest_float("gamma", 0, 10),
            reg_alpha=trial.suggest_float("reg_alpha", 0.000001, 10, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 0.000001, 10, log=True),
            scale_pos_weight = trial.suggest_float("scale_pos_weight", 1.0, 3.0)
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        scores = cross_val_score(
            model,
            X,
            y,
            cv=cross_validator,
            scoring="roc_auc",
            n_jobs=-1
        )

        return scores.mean()
    
    return objective

Então vamos criar nossa função objetivo com o dataset inteiro.

In [17]:
label = LabelEncoder()

X = df.drop(["patologia"], axis="columns")
y = label.fit_transform(df["patologia"])

objective = create_objetive(X, y)

<class 'pandas.DataFrame'>
Index: 11705 entries, 0 to 12872
Data columns (total 29 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   numeric_features__peso              11705 non-null  float64
 1   numeric_features__altura            11705 non-null  float64
 2   numeric_features__imc               11705 non-null  float64
 3   numeric_features__idade             11705 non-null  float64
 4   numeric_features__pa_sistolica      11705 non-null  float64
 5   numeric_features__pa_diastolica     11705 non-null  float64
 6   numeric_features__fc                11705 non-null  float64
 7   numeric_features__nascimento_dia    11705 non-null  float64
 8   numeric_features__nascimento_mes    11705 non-null  float64
 9   numeric_features__nascimento_ano    11705 non-null  float64
 10  numeric_features__atendimento_dia   11705 non-null  float64
 11  numeric_features__atendimento_mes   11705 non-null  float

E agora vamos otimizar os hiperparâmetros.

In [18]:
import optuna

study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.PercentilePruner(25.0, n_startup_trials=5)
)

study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True,
    n_jobs=-1
)

[I 2026-06-15 01:56:11,184] A new study created in memory with name: no-name-a67e6b34-ca47-4d71-8ca3-57dca5492722
Best trial: 0. Best value: 0.950075:   1%|          | 1/100 [00:06<10:02,  6.08s/it]

[I 2026-06-15 01:56:17,266] Trial 0 finished with value: 0.9500745026184271 and parameters: {'n_estimators': 405, 'learning_rate': 0.20585397211797046, 'max_depth': 3, 'min_child_weight': 13, 'subsample': 0.5444888008832027, 'colsample_bytree': 0.6844235108076275, 'gamma': 7.246975172372911, 'reg_alpha': 6.036318587335454, 'reg_lambda': 0.27046539183661883, 'scale_pos_weight': 1.919001673868314}. Best is trial 0 with value: 0.9500745026184271.


Best trial: 3. Best value: 0.952797:   2%|▏         | 2/100 [00:12<09:51,  6.03s/it]

[I 2026-06-15 01:56:23,254] Trial 3 finished with value: 0.9527970240799636 and parameters: {'n_estimators': 1779, 'learning_rate': 0.010967683419165822, 'max_depth': 7, 'min_child_weight': 7, 'subsample': 0.6029678947938029, 'colsample_bytree': 0.5838748912666333, 'gamma': 9.713747095132563, 'reg_alpha': 0.1905098098485435, 'reg_lambda': 0.0024663712298538735, 'scale_pos_weight': 2.570666045051265}. Best is trial 3 with value: 0.9527970240799636.


Best trial: 3. Best value: 0.952797:   3%|▎         | 3/100 [00:13<06:13,  3.85s/it]

...

[I 2026-06-15 02:08:54,633] Trial 97 finished with value: 0.9552133898467204 and parameters: {'n_estimators': 1860, 'learning_rate': 0.0016347057764273083, 'max_depth': 10, 'min_child_weight': 5, 'subsample': 0.6974323377736734, 'colsample_bytree': 0.9371633130729841, 'gamma': 0.03186267341883664, 'reg_alpha': 0.007533097458031189, 'reg_lambda': 7.651219573981398e-05, 'scale_pos_weight': 1.3326726136161082}. Best is trial 65 with value: 0.9553319812764534.


Best trial: 65. Best value: 0.955332:  99%|█████████▉| 99/100 [12:54<00:15, 15.08s/it]

[I 2026-06-15 02:09:05,846] Trial 98 finished with value: 0.9547909592529711 and parameters: {'n_estimators': 1596, 'learning_rate': 0.001497450710784081, 'max_depth': 9, 'min_child_weight': 5, 'subsample': 0.7072392917530483, 'colsample_bytree': 0.9112438164535632, 'gamma': 0.6120278595080761, 'reg_alpha': 0.056802426430621306, 'reg_lambda': 0.09346316450979275, 'scale_pos_weight': 1.5056175599852666}. Best is trial 65 with value: 0.9553319812764534.


Best trial: 65. Best value: 0.955332: 100%|██████████| 100/100 [13:10<00:00,  7.90s/it]

[I 2026-06-15 02:09:21,302] Trial 99 finished with value: 0.9548273512039982 and parameters: {'n_estimators': 1602, 'learning_rate': 0.0015021197195264552, 'max_depth': 9, 'min_child_weight': 5, 'subsample': 0.6994820491402198, 'colsample_bytree': 0.9100697635719157, 'gamma': 0.6456049108913098, 'reg_alpha': 0.007281766041722024, 'reg_lambda': 0.050157738278806874, 'scale_pos_weight': 1.1771260247945687}. Best is trial 65 with value: 0.9553319812764534.


Agora vamos montar o pipeline do nosso modelo ótimo.

In [24]:
params = study.best_params

model = XGBClassifier(
    **params,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=random_state,
    tree_method="hist",
    enable_categorical=True,
)

preprocessor = create_preprocessor(X)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

E vamos avaliar ele.

In [25]:
scores = cross_validate(
    model,
    X,
    y,
    cv=cross_validator,
    scoring={
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    n_jobs=-1,
)

In [26]:
for metric, values in scores.items():
    print(f"{metric:14} = {(100*values.mean()).round(2)}")

fit_time       = 961.96
score_time     = 65.79
test_roc_auc   = 95.53
test_accuracy  = 92.76
test_precision = 94.79
test_recall    = 87.74
test_f1        = 91.12


Como podemos notar, as métricas são tão excelentes quanto nos modelos anteriores. Vamos então analisar a importância das nossas features.

In [28]:
model.fit(X, y)

feature_names = X.columns
feature_importances = model.feature_importances_

importances = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances,
}).sort_values(by="importance", ascending=False)

print(importances.to_string())

                   feature  importance
9                    sopro    0.859900
8                       b2    0.036589
15                 motivo2    0.009138
4                   pulsos    0.008010
10                      fc    0.005133
17           hda1_faltante    0.005024
11                    hda1    0.004377
14                 motivo1    0.004254
5             pa_sistolica    0.003945
12                    hda2    0.003693
7                      ppa    0.003630
21       convenio_faltante    0.003557
28        dias_atendimento    0.003479
3                    idade    0.003445
1                   altura    0.003412
0                     peso    0.003373
24          nascimento_ano    0.003278
27         atendimento_ano    0.003220
6            pa_diastolica    0.003205
2                      imc    0.003187
26         atendimento_mes    0.003117
18         altura_faltante    0.003106
25         atendimento_dia    0.003070
16        motivo2_faltante    0.003026
23          nascimento_me

Como podemos notar, a feature mais importante continua sendo `"sopro"` com valor `"sistolico"` e ela é disparada a feature mais importante.

Agora, diferente do que fizemos nos modelos anteriores, vamos refazer a otimização de hiperparametros mas sem a variável `"sopro"`. A motivação para isso é o fato de que modelos baseados em Boosting com o XGBoost, LightGBM e CatBoost são muito poderoso e talvez eles encontrem alguma relação entre as features sem `"sopro"` e o target que os outros modelos não encontraram.

In [29]:
objective_sem_sopro = create_objetive(X.drop("sopro", axis="columns"), y)

study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.PercentilePruner(25.0, n_startup_trials=5)
)

study.optimize(
    objective_sem_sopro,
    n_trials=100,
    show_progress_bar=True,
    n_jobs=-1
)

[I 2026-06-13 16:28:41,681] A new study created in memory with name: no-name-4c54104c-f463-4e00-b1ee-3df3a9c07f20
  0%|          | 0/100 [00:00<?, ?it/s]/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 2. Best value: 0.808122:   1%|          | 1/100 [00:04<07:17,  4.41s/it]

[I 2026-06-13 16:28:46,089] Trial 2 finished with value: 0.808121756856931 and parameters: {'n_estimators': 947, 'learning_rate': 0.05133564064399223, 'max_depth': 6, 'min_child_weight': 17, 'subsample': 0.6726689330451595, 'colsample_bytree': 0.9982749261336382, 'gamma': 7.655383897216025, 'reg_alpha': 0.013366866159396874, 'reg_lambda': 1.1689394382473777e-05, 'scale_pos_weight': 1.576491964116663}. Best is trial 2 with value: 0.808121756856931.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 3. Best value: 0.808395:   2%|▏         | 2/100 [00:10<08:49,  5.40s/it]

[I 2026-06-13 16:28:52,180] Trial 3 finished with value: 0.8083945096726369 and parameters: {'n_estimators': 1959, 'learning_rate': 0.09285514312573959, 'max_depth': 7, 'min_child_weight': 14, 'subsample': 0.8249645473219476, 'colsample_bytree': 0.8652697575559887, 'gamma': 9.358653112961772, 'reg_alpha': 0.004081113558767837, 'reg_lambda': 0.0041138096947447396, 'scale_pos_weight': 1.6921588644376313}. Best is trial 3 with value: 0.8083945096726369.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 3. Best value: 0.808395:   3%|▎         | 3/100 [00:13<07:09,  4.43s/it]

...

[I 2026-06-13 16:33:04,095] Trial 97 finished with value: 0.8088693237523614 and parameters: {'n_estimators': 1750, 'learning_rate': 0.06290577503452827, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.8158152140226516, 'colsample_bytree': 0.6196191502028081, 'gamma': 4.279630605017196, 'reg_alpha': 0.008835339451940219, 'reg_lambda': 0.5355643250865622, 'scale_pos_weight': 1.9637189113752074}. Best is trial 57 with value: 0.8113764078528899.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 57. Best value: 0.811376:  99%|█████████▉| 99/100 [04:26<00:03,  3.14s/it]

[I 2026-06-13 16:33:08,173] Trial 98 finished with value: 0.8110594167722806 and parameters: {'n_estimators': 1740, 'learning_rate': 0.006954407666796061, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.8213502410315787, 'colsample_bytree': 0.6514432913030896, 'gamma': 4.319998651952965, 'reg_alpha': 0.007730964377156366, 'reg_lambda': 1.1099789506227005, 'scale_pos_weight': 1.9717956749950338}. Best is trial 57 with value: 0.8113764078528899.


Best trial: 57. Best value: 0.811376: 100%|██████████| 100/100 [04:28<00:00,  2.68s/it]

[I 2026-06-13 16:33:09,707] Trial 99 finished with value: 0.8046562268346923 and parameters: {'n_estimators': 1759, 'learning_rate': 0.1631254591009625, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.8919106141407036, 'colsample_bytree': 0.5883580155966659, 'gamma': 3.2602614891061723, 'reg_alpha': 0.008423678247301981, 'reg_lambda': 1.184994289341145, 'scale_pos_weight': 2.1523341327477286}. Best is trial 57 with value: 0.8113764078528899.


Vamos montar nosso pipeline.

In [ ]:
params_sem_sopro = study.best_params

model_sem_sopro = XGBClassifier(
    **params,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=random_state,
    tree_method="hist",
)

preprocessor_sem_sopro = create_preprocessor(X.drop("sopro", axis="columns"))

pipeline_sem_sopro = Pipeline(steps=[
    ("preprocessor", preprocessor_sem_sopro),
    ("model", model_sem_sopro)
])

E calcular as métricas de desempenho.

In [33]:
scores_sem_sopro = cross_validate(
    pipeline_sem_sopro,
    X.drop("sopro", axis="columns"),
    y,
    cv=cross_validator,
    scoring={
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    n_jobs=-1,
)

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [34]:
for metric, values in scores_sem_sopro.items():
    print(f"{metric:14} = {(100*values.mean()).round(2)}")

fit_time       = 1166.9
score_time     = 43.62
test_roc_auc   = 80.9
test_accuracy  = 73.79
test_precision = 68.46
test_recall    = 70.77
test_f1        = 69.59


Como podemos ver, as métricas sem a feature `"sopro"` continuam baixas como nos demais modelos, mas tivemos uma melhoria no recall.